# Assignment 5: Quicksort Algorithm Implementation, Analysis, and Randomization

**Name:** Sai Venkata Bharath Reddy Singareddy
**Course:** Algorithms and Data Structures  

## Purpose
In this assignment, I implement and study two versions of Quicksort in Python:

1. **Deterministic Quicksort**  
2. **Randomized Quicksort**

The main goal is to understand how Quicksort works, why it is usually efficient, when it performs poorly, and how randomization helps reduce the chance of worst-case behavior.


## Assignment Roadmap

This notebook is organized into the following parts:

- a short explanation of the Quicksort idea  
- deterministic Quicksort implementation  
- theoretical performance analysis  
- randomized Quicksort implementation  
- correctness checks  
- empirical timing comparison on different input types  
- discussion of results and conclusion  

I kept the explanation in separate markdown sections so the code stays easier to read.


## 1. Core Idea of Quicksort

Quicksort is a **divide-and-conquer** sorting algorithm.

At a high level, it works like this:

1. Choose a **pivot**
2. Rearrange the array so that:
   - elements less than or equal to the pivot are placed on the left
   - elements greater than the pivot are placed on the right
3. Recursively sort the left and right parts

The algorithm is efficient when the pivot creates reasonably balanced partitions. If the partitions become very uneven again and again, Quicksort becomes much slower.


In [1]:
import random
import time
import statistics
import sys
from typing import List, Callable

import pandas as pd

# Raise recursion limit slightly so the notebook can handle larger tests safely.
sys.setrecursionlimit(10000)


## 2. Partition Function

The partition function is the most important part of Quicksort.

In this implementation, the **last element** is used as the pivot. The function then scans the subarray and moves smaller values to the left side. At the end, it places the pivot in its correct final position.

After partitioning:
- everything on the left of the pivot is smaller than or equal to it
- everything on the right is greater than it


In [2]:
def partition(arr: List[int], low: int, high: int) -> int:
    pivot = arr[high]
    i = low - 1

    for j in range(low, high):
        if arr[j] <= pivot:
            i += 1
            arr[i], arr[j] = arr[j], arr[i]

    arr[i + 1], arr[high] = arr[high], arr[i + 1]
    return i + 1


## 3. Deterministic Quicksort

This version of Quicksort is called **deterministic** because the pivot rule is fixed.  
Every time, the last element is chosen as the pivot.

This is simple and easy to follow, but it can perform badly on already sorted or reverse-sorted input. In those cases, the partitions can become extremely unbalanced.


In [3]:
def quicksort(arr: List[int], low: int = 0, high: int = None) -> List[int]:
    if high is None:
        high = len(arr) - 1

    if low < high:
        pivot_index = partition(arr, low, high)
        quicksort(arr, low, pivot_index - 1)
        quicksort(arr, pivot_index + 1, high)

    return arr


## 4. Small Example for Deterministic Quicksort

Before moving into analysis, it is helpful to test the implementation on a small example.


In [4]:
sample = [29, 10, 14, 37, 13]
print("Original list:", sample)
print("Sorted list:  ", quicksort(sample.copy()))


Original list: [29, 10, 14, 37, 13]
Sorted list:   [10, 13, 14, 29, 37]


## 5. Theoretical Analysis of Deterministic Quicksort

### Best Case
The best case happens when the pivot splits the array into two nearly equal parts at every step.

- Work per level of recursion: about **O(n)**
- Number of levels: about **O(log n)**

So the best-case running time is:

\[
O(n \log n)
\]

### Average Case
On average, Quicksort tends to produce reasonably balanced splits, even if they are not perfectly equal.

That is why the **average-case** running time is also:

\[
O(n \log n)
\]

### Worst Case
The worst case happens when the pivot produces very uneven partitions repeatedly, such as sizes:

- \(n-1\) and \(0\)

This often happens when the input is already sorted and the pivot rule is fixed.

Then the recurrence becomes:

\[
T(n) = T(n-1) + O(n)
\]

which leads to:

\[
O(n^2)
\]

### Space Complexity
Quicksort is considered an in-place algorithm because it does not need an extra array of size \(n\).  
However, recursion uses call stack space:

- best/average case: **O(log n)**
- worst case: **O(n)**


## 6. Why Randomization Helps

A fixed pivot rule can be unlucky for certain inputs.  
To make Quicksort more reliable, we can choose the pivot **randomly**.

The idea of randomized Quicksort is simple:

- pick a random index from the current subarray
- swap that element with the last element
- use the normal partition process

This does **not remove** the worst case in theory, but it makes that bad case much less likely in practice.


In [5]:
def randomized_partition(arr: List[int], low: int, high: int) -> int:
    random_index = random.randint(low, high)
    arr[random_index], arr[high] = arr[high], arr[random_index]
    return partition(arr, low, high)


In [6]:
def randomized_quicksort(arr: List[int], low: int = 0, high: int = None) -> List[int]:
    if high is None:
        high = len(arr) - 1

    if low < high:
        pivot_index = randomized_partition(arr, low, high)
        randomized_quicksort(arr, low, pivot_index - 1)
        randomized_quicksort(arr, pivot_index + 1, high)

    return arr


## 7. Small Example for Randomized Quicksort

This example checks that the randomized version also sorts correctly.


In [7]:
sample = [29, 10, 14, 37, 13]
print("Original list:", sample)
print("Sorted list:  ", randomized_quicksort(sample.copy()))


Original list: [29, 10, 14, 37, 13]
Sorted list:   [10, 13, 14, 29, 37]


## 8. Time Complexity of Randomized Quicksort

The worst-case time complexity is still:

\[
O(n^2)
\]

because it is still possible, though unlikely, to keep choosing poor pivots.

However, the **expected** running time becomes:

\[
O(n \log n)
\]

This is the main advantage of randomization. It makes Quicksort much less sensitive to input order, so it usually behaves more consistently across random, sorted, and reverse-sorted data.


## 9. Correctness Checks

Before doing timing experiments, it is important to confirm that both implementations actually sort correctly.

The tests below compare the output of both algorithms with Python's built-in `sorted()` function on several different input cases.


In [8]:
test_cases = [
    [],
    [5],
    [3, 1, 2],
    [5, 5, 5, 5],
    [10, -1, 7, 3, 2, 8],
    list(range(10)),
    list(range(10, 0, -1))
]

for case in test_cases:
    assert quicksort(case.copy()) == sorted(case)
    assert randomized_quicksort(case.copy()) == sorted(case)

print("All correctness checks passed.")


All correctness checks passed.


## 10. Setup for Empirical Analysis

The assignment asks for an empirical comparison of deterministic and randomized Quicksort on different input sizes and distributions.

I use these three input types:

1. **Random input**
2. **Sorted input**
3. **Reverse-sorted input**

These are useful because they show how strongly pivot selection affects performance.


In [9]:
def generate_inputs(size: int):
    random_data = [random.randint(1, 100000) for _ in range(size)]
    sorted_data = list(range(size))
    reverse_sorted_data = list(range(size, 0, -1))
    return random_data, sorted_data, reverse_sorted_data


In [10]:
def measure_time(sort_function: Callable[[List[int]], List[int]], arr: List[int], repeats: int = 5) -> float:
    times = []
    for _ in range(repeats):
        data = arr.copy()
        start = time.perf_counter()
        sort_function(data)
        end = time.perf_counter()
        times.append(end - start)
    return statistics.mean(times)


## 11. Running the Timing Experiment

To keep the experiment fair, each test is repeated several times and the average time is recorded.

The input sizes are kept moderate because deterministic Quicksort can become slow on sorted and reverse-sorted input.


In [11]:
sizes = [100, 500, 1000, 1500]
results = []

for size in sizes:
    random_data, sorted_data, reverse_sorted_data = generate_inputs(size)

    results.append({
        "Input Size": size,
        "Deterministic - Random": measure_time(quicksort, random_data),
        "Deterministic - Sorted": measure_time(quicksort, sorted_data),
        "Deterministic - Reverse": measure_time(quicksort, reverse_sorted_data),
        "Randomized - Random": measure_time(randomized_quicksort, random_data),
        "Randomized - Sorted": measure_time(randomized_quicksort, sorted_data),
        "Randomized - Reverse": measure_time(randomized_quicksort, reverse_sorted_data),
    })

results_df = pd.DataFrame(results)
results_df


,Input Size,Deterministic - Random,Deterministic - Sorted,Deterministic - Reverse,Randomized - Random,Randomized - Sorted,Randomized - Reverse
0,100,0.000118,0.000862,0.000635,0.000189,0.000181,0.000220
1,500,0.001148,0.019476,0.009129,0.000699,0.000679,0.000679
2,1000,0.001180,0.088333,0.064839,0.002119,0.001837,0.002260
3,1500,0.003420,0.281582,0.187741,0.009884,0.003982,0.009163


## 12. Making the Results Easier to Read

The table below shows the average running times in seconds.  
Formatting the values makes the comparison easier to interpret.


In [12]:
formatted_df = results_df.copy()
for col in formatted_df.columns[1:]:
    formatted_df[col] = formatted_df[col].map(lambda x: f"{x:.6f}")

formatted_df


,Input Size,Deterministic - Random,Deterministic - Sorted,Deterministic - Reverse,Randomized - Random,Randomized - Sorted,Randomized - Reverse
0,100,0.000118,0.000862,0.000635,0.000189,0.000181,0.000220
1,500,0.001148,0.019476,0.009129,0.000699,0.000679,0.000679
2,1000,0.001180,0.088333,0.064839,0.002119,0.001837,0.002260
3,1500,0.003420,0.281582,0.187741,0.009884,0.003982,0.009163


## 13. Discussion of the Results

The expected pattern is:

- On **random input**, both versions should usually perform close to \(O(n \log n)\)
- On **sorted** and **reverse-sorted input**, deterministic Quicksort may slow down significantly
- Randomized Quicksort should usually remain more stable across all input types

If the table shows that deterministic Quicksort takes noticeably longer on sorted or reverse-sorted data, that matches the theoretical analysis.

If randomized Quicksort stays more consistent, that shows the benefit of random pivot selection.


## 14. Interpretation in Simple Terms

From a practical point of view, the main lesson is this:

- **Deterministic Quicksort** can be very fast, but it depends heavily on the input order
- **Randomized Quicksort** reduces the risk of bad pivot choices, so it is usually safer and more consistent

This is why randomization is often used in algorithm design. It does not guarantee perfection, but it makes poor behavior much less likely.


## 15. Conclusion

In this assignment, I implemented both deterministic and randomized Quicksort in Python and compared them both theoretically and empirically.

The deterministic version works well in many cases, but its performance can degrade badly when the pivot rule interacts poorly with the input order. The randomized version improves robustness by choosing pivots at random, which usually leads to more balanced partitions.

Overall, the assignment showed that:

- Quicksort is efficient in the best and average cases
- its worst case is much slower
- randomization helps reduce the chance of worst-case behavior in practice

This makes randomized Quicksort a more dependable version for real use.
